In [4]:
import MetaTrader5 as mt5
import pandas as pd
import time
import pytz
from datetime import datetime
import numpy as np

mt5.initialize()


def get_values(symbol):
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_H4, 0, 200)
    rates_frame = pd.DataFrame(rates)

    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame['rsi'] = get_rsi(rates_frame['close'], 21)
    rates_frame['sma']= rates_frame['close'].rolling(window=100).mean()

    return rates_frame

def get_rsi(close, lookback):
#     t = time.time()
    ret = close.diff()
    
    up = []
    down = []
    for i in range(len(ret)):
        if ret[i] < 0:
            up.append(0)
            down.append(ret[i])
        else:
            up.append(ret[i])
            down.append(0)
    up_series = pd.Series(up)
    down_series = pd.Series(down).abs()
    up_ewm = up_series.ewm(com = lookback - 1, adjust = False).mean()
    down_ewm = down_series.ewm(com = lookback - 1, adjust = False).mean()
    rs = up_ewm/down_ewm
    rsi = 100 - (100 / (1 + rs))
    rsi_df = pd.DataFrame(rsi).rename(columns = {0:'rsi'}).set_index(close.index)
#     print(time.time()-t)
    return rsi_df

def Action_close(ticket_no, symbol, signal, lot):
    try:
        a = [[mt5.symbol_info_tick(symbol).ask, mt5.ORDER_TYPE_BUY], [mt5.symbol_info_tick(symbol).bid, mt5.ORDER_TYPE_SELL]]
        position_id=ticket_no
        price = a[signal][0]
        deviation=1000
        request={
            "action": mt5.TRADE_ACTION_DEAL,    
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][1],
            "position": position_id,
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script close",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result=mt5.order_send(request)
        return result
    except Exception as e:
        print("Action_close_Error")
        print(e)

def Action(symbol, lot, signal):
    try:
        symbol_info = mt5.symbol_info(symbol)

        a = [[mt5.ORDER_TYPE_SELL, mt5.symbol_info_tick(symbol).bid], [mt5.ORDER_TYPE_BUY, mt5.symbol_info_tick(symbol).ask]]
        price = a[signal][1]
        deviation = 200
        
        request = {
            "action": mt5.TRADE_ACTION_DEAL,
            "symbol": symbol,
            "volume": lot,
            "type": a[signal][0],
            "price": price,
            "deviation": deviation,
            "magic": 234000,
            "comment": "python script open",
            "type_time": mt5.ORDER_TIME_GTC,
            "type_filling": mt5.ORDER_FILLING_FOK,
        }
        result = mt5.order_send(request)
        return result
    except Exception as e:
        print("Action")
        print(e)


def run(symbol):
    check = 0
    lot = 0.1
    buy_check = 0
    sell_check = 0
    buy_up = 0
    sell_up = 0
    order_time = 0
    old = 0
    old_pp = 0

    buy = 1
    sell = 0
 
    print(symbol)
    hour_passed = True

    while True:
        a = get_values(symbol)
        if a.iloc[-2].close != old:
            try:
                pp = mt5.positions_get(ticket=result_buy.order)[0].profit
                old_pp= pp
            except:
                pass
            if a.iloc[-2].rsi1 <= 50 and a.iloc[-3].rsi1 <= 50 and a.iloc[-4].rsi1 <= 50 and a.iloc[-2].rsi1 >= 29 and \
                a.iloc[-3].rsi1 >= 30 and a.iloc[-4].rsi1 > 30 and a.iloc[-5].rsi1 >= 50:
                if a.iloc[-6].rsi1 >= 70 :
                    if (a.iloc[-5].open - a.iloc[-5].close) <0.250 and (a.iloc[-4].open - a.iloc[-4].close) <0.250:             
                        print("+++"*20)

                        print(f"{a.iloc[i].name} -- ")
                        buy_price = a.iloc[i].close
                        check=1
                    else:
                        pass
                else:
                    print("=="*20)

                    print(f"{a.iloc[i].name} -- ")
                    buy_price = a.iloc[i].close
                    check=1
            

#             if  a.iloc[-2].rsi <= 29.9 and  a.iloc[-3].rsi >= 29.9 and a.iloc[-2].close < a.iloc[-2].sma and sell_check == 0:   

#                 result_sell = Action(symbol, lot, sell)
#                 print(f"Symbol-->{symbol} ||| Type-->Sell  ||| Ticket_No-->{result_sell.order}")

#                 sell_check = 1

#                 buy_check = 0

            if a.iloc[-2].rsi >= 68 and  a.iloc[-3].rsi <= 68 and  a.iloc[-2].close > a.iloc[-2].sma and buy_check==0:

                result_buy = Action(symbol, lot, buy)
                print(f"Symbol-->{symbol} ||| Type-->Buy ||| Ticket_No-->{result_buy.order} ||| result_comment-->{result_buy.comment}")
                buy_check = 1
                old = a.iloc[-2].close

                sell_check = 0

        if sell_check==1:
            pp = mt5.positions_get(ticket=result_sell.order)[0].profit
            if pp<= -10.0 and sell_check == 1:
                result_sell = Action_close(result_sell.order, symbol, sell, lot)   #Action_close

                if result_sell.comment == "Requote":
                    result_sell = Action_close(result_sell.order, symbol, sell, lot)
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment}")
                sell_check = 0
            if pp >= 10.0 and sell_check == 1:
                result_sell = Action_close(result_sell.order, symbol, sell, lot)   #Action_close


                if result_sell.comment == "Requote":
                    result_sell = Action_close(result_sell.order, symbol, sell, lot)
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment} ||| Requoted")
                else:
                    print(f"Close  Symbol-->{symbol} ||| Type-->Sell ||| result_comment-->{result_sell.comment}")
                sell_check = 0

        if buy_check==1:
            pp = mt5.positions_get(ticket=result_buy.order)[0].profit
            if a.iloc[-1].rsi < a.iloc[-2].rsi and buy_check == 1:
                diff = abs(a.iloc[-2].close - a.iloc[-2].open)/2
                if pp <= 0.0 and a.iloc[-1].close < (a.iloc[-1].open-diff):
                    result_buy = Action_close(result_buy.order, symbol, buy, lot)     #Action_close

                    if result_buy.comment == "Requote":
                        result_buy = Action_close(result_buy.order, symbol, buy, lot)
                        print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment} ||| Requoted")
                    else:
                        print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment}")  
                    buy_check = 0
                elif pp/2 <= old_pp:
                    result_buy = Action_close(result_buy.order, symbol, buy, lot)     #Action_close

                    if result_buy.comment == "Requote":
                        result_buy = Action_close(result_buy.order, symbol, buy, lot)
                        print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment} ||| Requoted")
                    else:
                        print(f"Close  Symbol-->{symbol} ||| Type-->Buy ||| result_comment-->{result_buy.comment}")  
                    buy_check = 0
#         if 

#             hour_passed = True
        time.sleep(1)

for symbol in ['USDJPY']:
    run(symbol)
    

USDJPY


KeyboardInterrupt: 

In [13]:
def get_values(symbol):
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_H4, 0, 200)
    rates_frame = pd.DataFrame(rates)
    rates_frame['time']=pd.to_datetime(rates_frame['time'], unit='s')
    rates_frame = rates_frame.set_index('time')
    rates_frame['rsi'] = get_rsi(rates_frame['close'], 21)
    rates_frame['sma']= rates_frame['close'].rolling(window=100).mean()

#     return rates_frame


In [14]:
get_values('USDJPY')